In [9]:
# Student Identity
LAST_NAME = "Asis"
STUDENT_ID = "TUPM-26-1217"
FAVORITE_ARTIST = "EIICHIRO ODA"
SEED_NUM = 7

# Derive computational parameters from the fixed inputs
seed_digit = int(STUDENT_ID[-1])
id_checksum = sum(int(d) for d in STUDENT_ID if d.isdigit())
vector_dim = len(LAST_NAME)
artist_positions = [ord(c) - 64 for c in FAVORITE_ARTIST if c.isalpha()]

# Encapsulate state into a Dictionary
sys_config = {
    "operator": LAST_NAME,
    "artist": FAVORITE_ARTIST,
    "authid": seed_digit,
    "baseseed": SEED_NUM,
    "checksum": id_checksum,
    "vectordim": vector_dim,
    "positions": artist_positions,
    "status": "INITIALIZED"
}

print("=== SYSTEM CONFIGURATION ===")
for key, value in sys_config.items():
    print(f"{key.upper()}: {value}")

=== SYSTEM CONFIGURATION ===
OPERATOR: Asis
ARTIST: EIICHIRO ODA
AUTHID: 7
BASESEED: 7
CHECKSUM: 19
VECTORDIM: 4
POSITIONS: [5, 9, 9, 3, 8, 9, 18, 15, 15, 4, 1]
STATUS: INITIALIZED


In [10]:
sys_config = {
    "operator": LAST_NAME,
    "artist": FAVORITE_ARTIST,
    "baseseed": SEED_NUM,
    "checksum": id_checksum,
    "vectordim": vector_dim,
    "positions": artist_positions,
    "status": "INITIALIZED"
}
print("=== SYSTEM CONFIGURATION ===")
for key, value in sys_config.items():
    print(f"{key.upper()}: {value}")


=== SYSTEM CONFIGURATION ===
OPERATOR: Asis
ARTIST: EIICHIRO ODA
BASESEED: 7
CHECKSUM: 19
VECTORDIM: 4
POSITIONS: [5, 9, 9, 3, 8, 9, 18, 15, 15, 4, 1]
STATUS: INITIALIZED


In [11]:
base = sys_config["baseseed"]
positions = sys_config["positions"]

customers = [f"{sys_config['operator']}_C{i}" for i in range(1, 6)]
products = ["NOTEBOOK", "PEN", "USB", "HEADPHONES", "CALCULATOR"]
prices = {"NOTEBOOK": 59.00, "PEN": 15.50, "USB": 320.00,
          "HEADPHONES": 899.00, "CALCULATOR": 450.00}

transactions = []
for row, cust in enumerate(customers):
    for col, prod in enumerate(products):
        if (row + col) % 3 == 0:          # not every customer buys everything
            qty = 1 + (base * (row + 1) + positions[(row + col) % len(positions)]) % 4
            transactions.append((cust, prod, qty, prices[prod]))

# At least one repeated customer-product transaction
transactions.append((customers[0], products[0], 2, prices[products[0]]))

print("=== GENERATED TRANSACTION RECORDS ===")
for t in transactions:
    print(t)
print(f"\nTotal records: {len(transactions)}")


=== GENERATED TRANSACTION RECORDS ===
('Asis_C1', 'NOTEBOOK', 1, 59.0)
('Asis_C1', 'HEADPHONES', 3, 899.0)
('Asis_C2', 'USB', 2, 320.0)
('Asis_C3', 'PEN', 1, 15.5)
('Asis_C3', 'CALCULATOR', 4, 450.0)
('Asis_C4', 'NOTEBOOK', 4, 59.0)
('Asis_C4', 'HEADPHONES', 3, 899.0)
('Asis_C5', 'USB', 2, 320.0)
('Asis_C1', 'NOTEBOOK', 2, 59.0)

Total records: 9


In [12]:
organized = {}
for cust, prod, qty, price in transactions:
    value = round(qty * price, 2)
    organized.setdefault(cust, []).append((prod, qty, price, value))

total_value = round(sum(v for cust in organized for *_, v in organized[cust]), 2)
overall = sorted({prod for _, prod, _, _ in transactions})
distinct_customers = sorted(organized)

print("=== ORGANIZED TRANSACTION DATA ===")
for cust in distinct_customers:
    print(f"\nCustomer: {cust}")
    for prod, qty, price, value in organized[cust]:
        print(f"  {prod:<12} qty={qty}  price={price:<7} value={value}")
print(f"\nOverall transaction value: {total_value}")
print(f"Distinct products : {len(overall)} -> {overall}")
print(f"Distinct customers: {len(distinct_customers)}")


=== ORGANIZED TRANSACTION DATA ===

Customer: Asis_C1
  NOTEBOOK     qty=1  price=59.0    value=59.0
  HEADPHONES   qty=3  price=899.0   value=2697.0
  NOTEBOOK     qty=2  price=59.0    value=118.0

Customer: Asis_C2
  USB          qty=2  price=320.0   value=640.0

Customer: Asis_C3
  PEN          qty=1  price=15.5    value=15.5
  CALCULATOR   qty=4  price=450.0   value=1800.0

Customer: Asis_C4
  NOTEBOOK     qty=4  price=59.0    value=236.0
  HEADPHONES   qty=3  price=899.0   value=2697.0

Customer: Asis_C5
  USB          qty=2  price=320.0   value=640.0

Overall transaction value: 8902.5
Distinct products : 5 -> ['CALCULATOR', 'HEADPHONES', 'NOTEBOOK', 'PEN', 'USB']
Distinct customers: 5


In [13]:
pair_list = [(cust, prod) for cust, prod, _, _ in transactions]
pair_set = set(pair_list)
repeated = [pair for pair in pair_set if pair_list.count(pair) > 1]

print("=== REPEATED / DISTINCT TRANSACTIONS ===")
print(f"Total records     : {len(pair_list)}")
print(f"Distinct purchases: {len(pair_set)}")
print(f"Repeated purchases: {repeated}")


=== REPEATED / DISTINCT TRANSACTIONS ===
Total records     : 9
Distinct purchases: 8
Repeated purchases: [('Asis_C1', 'NOTEBOOK')]


In [14]:
customer_summary = {}
product_summary = {}

for cust, prod, qty, price in transactions:
    value = round(qty * price, 2)
    customer_summary[cust] = round(customer_summary.get(cust, 0) + value, 2)

    prev_qty, prev_val = product_summary.get(prod, (0, 0))
    product_summary[prod] = (prev_qty + qty, round(prev_val + value, 2))

print("=== CUSTOMER SUMMARIES ===")
for cust in sorted(customer_summary, key=customer_summary.get, reverse=True):
    print(f"{cust:<20} total = {customer_summary[cust]}")

print("\n=== PRODUCT SUMMARIES ===")
for prod in sorted(product_summary, key=lambda p: product_summary[p][1], reverse=True):
    qty, val = product_summary[prod]
    print(f"{prod:<12} sold={qty:<4} revenue={val}")


=== CUSTOMER SUMMARIES ===
Asis_C4              total = 2933.0
Asis_C1              total = 2874.0
Asis_C3              total = 1815.5
Asis_C2              total = 640.0
Asis_C5              total = 640.0

=== PRODUCT SUMMARIES ===
HEADPHONES   sold=6    revenue=5394.0
CALCULATOR   sold=4    revenue=1800.0
USB          sold=4    revenue=1280.0
NOTEBOOK     sold=7    revenue=413.0
PEN          sold=1    revenue=15.5


In [15]:
top_customers = sorted(customer_summary.items(), key=lambda i: i[1], reverse=True)
top_products = sorted(product_summary.items(), key=lambda i: i[1][1], reverse=True)

print("=== ORDERED RESULTS ===")
print("Top customers by spend:")
for rank, (cust, total) in enumerate(top_customers, start=1):
    print(f"  {rank}. {cust:<20} {total}")
print("Top products by revenue:")
for rank, (prod, (qty, val)) in enumerate(top_products, start=1):
    print(f"  {rank}. {prod:<12} {val}")

print("\n=== FINAL CONSOLIDATED TRANSACTION REPORT ===")
print(f"Operator          : {sys_config['operator']}")
print(f"Records           : {len(transactions)}")
print(f"Customers         : {len(distinct_customers)}")
print(f"Products          : {len(overall)}")
print(f"Repeated purchases: {len(repeated)}")
print(f"Overall value     : {total_value}")
print(f"Top customer      : {top_customers[0][0]} ({top_customers[0][1]})")
print(f"Top product       : {top_products[0][0]} ({top_products[0][1][1]})")


=== ORDERED RESULTS ===
Top customers by spend:
  1. Asis_C4              2933.0
  2. Asis_C1              2874.0
  3. Asis_C3              1815.5
  4. Asis_C2              640.0
  5. Asis_C5              640.0
Top products by revenue:
  1. HEADPHONES   5394.0
  2. CALCULATOR   1800.0
  3. USB          1280.0
  4. NOTEBOOK     413.0
  5. PEN          15.5

=== FINAL CONSOLIDATED TRANSACTION REPORT ===
Operator          : Asis
Records           : 9
Customers         : 5
Products          : 5
Repeated purchases: 1
Overall value     : 8902.5
Top customer      : Asis_C4 (2933.0)
Top product       : HEADPHONES (5394.0)
